In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz

def show_revenue_installs_chart(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 13:00 (1 PM) and 14:00 (2 PM)
    if not (13 <= current_time_ist.hour < 14):
        print(f"Graph is currently hidden. It is only available between 1 PM and 2 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function, preventing the graph from rendering in the dashboard

    # 3. Load Data
    df = pd.read_csv(csv_file_path)

    # 4. Clean and Transform Columns
    # Clean App Name length (<= 30 characters)
    df = df[df['App'].astype(str).str.len() <= 30]

    # Clean Content Rating
    df = df[df['Content Rating'] == 'Everyone']

    # Clean Size (Extract numeric MB and filter > 15M)
    def clean_size_mb(size):
        if isinstance(size, str):
            if 'M' in size:
                return float(size.replace('M', ''))
            elif 'k' in size:
                return float(size.replace('k', '')) / 1024 # Convert kb to MB
        return np.nan
    
    df['Size_MB'] = df['Size'].apply(clean_size_mb)
    df = df[df['Size_MB'] > 15.0]

    # Clean Installs (Convert to numeric and filter >= 10,000)
    df['Installs'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df = df[df['Installs'] >= 10000]

    # Clean Price and Calculate Revenue
    df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Revenue'] = df['Price'] * df['Installs']
    
    # Filter Revenue >= $10,000
    # Note: This will naturally exclude 'Free' apps because their revenue is $0. 
    # If you want Free apps included in the comparison despite having $0 revenue, 
    # you would need to change this condition to: df[(df['Revenue'] >= 10000) | (df['Type'] == 'Free')]
    df = df[df['Revenue'] >= 10000]

    # Clean Android Ver (Extract major/minor version and filter > 4.0)
    def check_android_ver(ver):
        if pd.isna(ver) or ver == 'Varies with device':
            return False
        # Extract the first float-like sequence (e.g., "4.0.3 and up" -> 4.0)
        import re
        match = re.search(r'^(\d+\.\d+)', str(ver))
        if match:
            return float(match.group(1)) > 4.0
        return False
        
    df = df[df['Android Ver'].apply(check_android_ver)]

    # 5. Get the Top 3 Categories by total installs from the filtered data
    top_3_cats = df.groupby('Category')['Installs'].sum().nlargest(3).index
    df_top_3 = df[df['Category'].isin(top_3_cats)]

    # 6. Group by Category and Type to get average Installs and Revenue
    agg_df = df_top_3.groupby(['Category', 'Type']).agg({
        'Installs': 'mean',
        'Revenue': 'mean'
    }).reset_index()

    # 7. Render the Dual-Axis Chart
    # We will pivot the data to make plotting grouped bars easier
    pivot_installs = agg_df.pivot(index='Category', columns='Type', values='Installs').fillna(0)
    pivot_revenue = agg_df.pivot(index='Category', columns='Type', values='Revenue').fillna(0)

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()

    x = np.arange(len(pivot_installs.index))
    width = 0.2

    # Plot Installs on primary y-axis
    if 'Paid' in pivot_installs.columns:
        ax1.bar(x - width, pivot_installs['Paid'], width, label='Paid Avg Installs', color='lightblue')
    if 'Free' in pivot_installs.columns:
        ax1.bar(x, pivot_installs['Free'], width, label='Free Avg Installs', color='deepskyblue')

    # Plot Revenue on secondary y-axis
    if 'Paid' in pivot_revenue.columns:
        ax2.bar(x + width, pivot_revenue['Paid'], width, label='Paid Avg Revenue', color='lightgreen')
    if 'Free' in pivot_revenue.columns:
        ax2.bar(x + width*2, pivot_revenue['Free'], width, label='Free Avg Revenue', color='forestgreen')

    # Formatting
    ax1.set_xticks(x)
    ax1.set_xticklabels(pivot_installs.index)
    ax1.set_ylabel('Average Installs', color='tab:blue')
    ax2.set_ylabel('Average Revenue ($)', color='tab:green')
    ax1.set_title('Top 3 Categories: Avg Installs vs Avg Revenue (Paid vs Free)')

    # Combine legends
    lines_labels = [ax.get_legend_handles_labels() for ax in [ax1, ax2]]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    ax1.legend(lines, labels, loc='upper left')

    plt.tight_layout()
    plt.show()

# Usage
show_revenue_installs_chart('googleplaystore.csv')

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz

def show_revenue_installs_chart(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 13:00 (1 PM) and 14:00 (2 PM)
    if not (13 <= current_time_ist.hour < 14):
        print(f"Graph is currently hidden. It is only available between 1 PM and 2 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function, preventing the graph from rendering in the dashboard

    # 3. Load Data
    df = pd.read_csv(csv_file_path)

    # 4. Clean and Transform Columns
    # Clean App Name length (<= 30 characters)
    df = df[df['App'].astype(str).str.len() <= 30]

    # Clean Content Rating
    df = df[df['Content Rating'] == 'Everyone']

    # Clean Size (Extract numeric MB and filter > 15M)
    def clean_size_mb(size):
        if isinstance(size, str):
            if 'M' in size:
                return float(size.replace('M', ''))
            elif 'k' in size:
                return float(size.replace('k', '')) / 1024 # Convert kb to MB
        return np.nan
    
    df['Size_MB'] = df['Size'].apply(clean_size_mb)
    df = df[df['Size_MB'] > 15.0]

    # Clean Installs (Convert to numeric and filter >= 10,000)
    df['Installs'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df = df[df['Installs'] >= 10000]

    # Clean Price and Calculate Revenue
    df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Revenue'] = df['Price'] * df['Installs']
    
    # Filter Revenue >= $10,000
    # Note: This will naturally exclude 'Free' apps because their revenue is $0. 
    # If you want Free apps included in the comparison despite having $0 revenue, 
    # you would need to change this condition to: df[(df['Revenue'] >= 10000) | (df['Type'] == 'Free')]
    df = df[df['Revenue'] >= 10000]

    # Clean Android Ver (Extract major/minor version and filter > 4.0)
    def check_android_ver(ver):
        if pd.isna(ver) or ver == 'Varies with device':
            return False
        # Extract the first float-like sequence (e.g., "4.0.3 and up" -> 4.0)
        import re
        match = re.search(r'^(\d+\.\d+)', str(ver))
        if match:
            return float(match.group(1)) > 4.0
        return False
        
    df = df[df['Android Ver'].apply(check_android_ver)]

    # 5. Get the Top 3 Categories by total installs from the filtered data
    top_3_cats = df.groupby('Category')['Installs'].sum().nlargest(3).index
    df_top_3 = df[df['Category'].isin(top_3_cats)]

    # 6. Group by Category and Type to get average Installs and Revenue
    agg_df = df_top_3.groupby(['Category', 'Type']).agg({
        'Installs': 'mean',
        'Revenue': 'mean'
    }).reset_index()

    # 7. Render the Dual-Axis Chart
    # We will pivot the data to make plotting grouped bars easier
    pivot_installs = agg_df.pivot(index='Category', columns='Type', values='Installs').fillna(0)
    pivot_revenue = agg_df.pivot(index='Category', columns='Type', values='Revenue').fillna(0)

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()

    x = np.arange(len(pivot_installs.index))
    width = 0.2

    # Plot Installs on primary y-axis
    if 'Paid' in pivot_installs.columns:
        ax1.bar(x - width, pivot_installs['Paid'], width, label='Paid Avg Installs', color='lightblue')
    if 'Free' in pivot_installs.columns:
        ax1.bar(x, pivot_installs['Free'], width, label='Free Avg Installs', color='deepskyblue')

    # Plot Revenue on secondary y-axis
    if 'Paid' in pivot_revenue.columns:
        ax2.bar(x + width, pivot_revenue['Paid'], width, label='Paid Avg Revenue', color='lightgreen')
    if 'Free' in pivot_revenue.columns:
        ax2.bar(x + width*2, pivot_revenue['Free'], width, label='Free Avg Revenue', color='forestgreen')

    # Formatting
    ax1.set_xticks(x)
    ax1.set_xticklabels(pivot_installs.index)
    ax1.set_ylabel('Average Installs', color='tab:blue')
    ax2.set_ylabel('Average Revenue ($)', color='tab:green')
    ax1.set_title('Top 3 Categories: Avg Installs vs Avg Revenue (Paid vs Free)')

    # Combine legends
    lines_labels = [ax.get_legend_handles_labels() for ax in [ax1, ax2]]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    ax1.legend(lines, labels, loc='upper left')

    plt.tight_layout()
    plt.show()

# Usage
show_revenue_installs_chart('googleplaystore.csv')

Graph is currently hidden. It is only available between 1 PM and 2 PM IST. (Current time: 02:14 PM IST)
